# Step 5: Train and evaluate

RandomForest and XGBoost on the 32 histone features, predicting Stable vs Transient. Missing histone values are imputed with the training set's median only.

## Config

In [15]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import shap
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, roc_auc_score, confusion_matrix,
                              ConfusionMatrixDisplay, balanced_accuracy_score, f1_score)
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler

ROOT = Path("..").resolve()
PROC = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
SHAP_DIR = RESULTS / "figures" / "shap"
RESULTS.mkdir(exist_ok=True)
SHAP_DIR.mkdir(parents=True, exist_ok=True)

MODELING_CSV = PROC / "modeling_table.csv"

TARGET_NAMES = ["Transient (1-2)", "Stable (3-4)"]
N_ESTIMATORS = 400
SEED = 42

RF_PARAMS = {"max_depth": [8, 12, None], "min_samples_split": [2, 5],
             "class_weight": ["balanced", "balanced_subsample", None]}
XGB_PARAMS = {"max_depth": [3, 5, 8], "learning_rate": [0.01, 0.1, 0.2],
              "scale_pos_weight": [1, 2, 5]}

## Load, split features/target

In [16]:
df = pd.read_csv(MODELING_CSV)

non_feature_cols = ["gene_name", "target", "high_connectivity_count"] + \
    [c for c in df.columns if c.endswith("_is_hub")]
X = df.drop(columns=non_feature_cols)
y = df["target"]

print(f"X: {X.shape}, y distribution: {y.value_counts().to_dict()}")
print(f"NaNs per feature:\n{X.isna().sum()[X.isna().sum() > 0].to_string()}")

X: (4509, 32), y distribution: {1: 2865, 0: 1644}
NaNs per feature:
H3K4me1_C1-8       9
H3K4me1_C9-13      1
H3K4me1_C14E       1
H3K4me3_C1-8     243
H3K4me3_C9-13     19
H3K9ac_C1-8       16
H3K18ac_C1-8     202
H3K18ac_C14E       1
H3K27ac_C1-8      96
H3K27ac_C9-13      1


## Split, impute on train only, scale

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X.columns, index=X_test.index)

scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=X.columns, index=X_train.index)
X_test_s = pd.DataFrame(scaler.transform(X_test_imp), columns=X.columns, index=X_test.index)

print(f"train: {y_train.value_counts().to_dict()}, test: {y_test.value_counts().to_dict()}")

train: {1: 2292, 0: 1315}, test: {1: 573, 0: 329}


## Models and tuning

In [18]:
def make_models():
    rf = RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=SEED, n_jobs=-1)
    xgb_model = xgb.XGBClassifier(n_estimators=N_ESTIMATORS, random_state=SEED, n_jobs=-1,
                                   eval_metric="logloss", objective="binary:logistic")
    return [(rf, RF_PARAMS, "RANDOM FOREST"), (xgb_model, XGB_PARAMS, "XGBOOST")]

def tune_and_score(model, params, X_tr, y_tr, X_te, y_te):
    search = RandomizedSearchCV(model, param_distributions=params, n_iter=5, cv=3,
                                 scoring="balanced_accuracy", random_state=SEED, n_jobs=-1)
    search.fit(X_tr, y_tr)
    best = search.best_estimator_
    y_pred = best.predict(X_te)
    y_proba = best.predict_proba(X_te)[:, 1]
    metrics = {"roc_auc": roc_auc_score(y_te, y_proba),
               "balanced_accuracy": balanced_accuracy_score(y_te, y_pred),
               "f1_macro": f1_score(y_te, y_pred, average="macro"),
               "best_params": str(search.best_params_)}
    return best, y_pred, metrics

## Train, evaluate, confusion matrices

In [19]:
def save_figure(fig, path):
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"saved -> {path.relative_to(ROOT)}")

best_models, all_metrics = {}, []

for model, params, name in make_models():
    best, y_pred, metrics = tune_and_score(model, params, X_train_s, y_train, X_test_s, y_test)
    best_models[name] = best

    print(f"\n--- {name} ---")
    print(f"best params: {metrics['best_params']}")
    print(classification_report(y_test, y_pred, target_names=TARGET_NAMES))
    print(f"ROC AUC: {metrics['roc_auc']:.4f} | balanced acc: {metrics['balanced_accuracy']:.4f} "
          f"| macro F1: {metrics['f1_macro']:.4f}")
    print(confusion_matrix(y_test, y_pred))

    slug = name.lower().replace(" ", "_")
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=TARGET_NAMES,
                                             cmap="Blues", ax=axes[0], colorbar=False)
    axes[0].set_title("counts")
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=TARGET_NAMES,
                                             normalize="true", values_format=".2f",
                                             cmap="Blues", ax=axes[1], colorbar=False)
    axes[1].set_title("normalized")
    fig.suptitle(f"{name} confusion matrix")
    fig.tight_layout()
    save_figure(fig, RESULTS / "figures" / f"{slug}_confusion_matrix.png")

    all_metrics.append({"model": name, **metrics})


--- RANDOM FOREST ---
best params: {'min_samples_split': 5, 'max_depth': 12, 'class_weight': 'balanced'}
                 precision    recall  f1-score   support

Transient (1-2)       0.61      0.33      0.43       329
   Stable (3-4)       0.70      0.88      0.78       573

       accuracy                           0.68       902
      macro avg       0.65      0.60      0.60       902
   weighted avg       0.67      0.68      0.65       902

ROC AUC: 0.7081 | balanced acc: 0.6048 | macro F1: 0.6026
[[108 221]
 [ 68 505]]
saved -> results\figures\random_forest_confusion_matrix.png

--- XGBOOST ---
best params: {'scale_pos_weight': 1, 'max_depth': 5, 'learning_rate': 0.2}
                 precision    recall  f1-score   support

Transient (1-2)       0.56      0.37      0.45       329
   Stable (3-4)       0.70      0.83      0.76       573

       accuracy                           0.66       902
      macro avg       0.63      0.60      0.60       902
   weighted avg       0.65   

## Cross-validated scores

In [20]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []

for name, best in best_models.items():
    pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", clone(best))
    ])
    s = cross_validate(
        pipe, X, y, cv=cv, scoring=["roc_auc", "balanced_accuracy", "f1_macro"]
    )
    row = {"model": name}
    for m in ["roc_auc", "balanced_accuracy", "f1_macro"]:
        row[f"cv_{m}"] = s[f"test_{m}"].mean()
        row[f"cv_{m}_sd"] = s[f"test_{m}"].std()
    cv_rows.append(row)
    print(
        f"{name}: AUC {row['cv_roc_auc']:.4f} +- {row['cv_roc_auc_sd']:.3f} | "
        f"bal acc {row['cv_balanced_accuracy']:.4f} +- {row['cv_balanced_accuracy_sd']:.3f} | "
        f"macro F1 {row['cv_f1_macro']:.4f} +- {row['cv_f1_macro_sd']:.3f}"
    )

cv_summary = pd.DataFrame(cv_rows)
cv_summary.to_csv(RESULTS / "results_cv.csv", index=False)

RANDOM FOREST: AUC 0.7022 +- 0.018 | bal acc 0.5971 +- 0.010 | macro F1 0.5919 +- 0.012
XGBOOST: AUC 0.6909 +- 0.031 | bal acc 0.6184 +- 0.025 | macro F1 0.6208 +- 0.028


## Hub cutoff sensitivity

In [21]:
EIGEN_CSV = PROC / "eigenvector_features.csv"
HISTONE_CSV = PROC / "histone_features.csv"
STAGES = ["C1-8", "C9-13", "C14E", "C14L"]

eigen_s = pd.read_csv(EIGEN_CSV)
hist_s = pd.read_csv(HISTONE_CSV)

sens_rows = []

for q in [0.80, 0.85, 0.90, 0.95]:
    e = eigen_s.copy()
    e["cnt"] = 0

    for st in STAGES:
        cut = e.groupby("chrom")[f"centrality_{st}"].transform(
            lambda s: s.quantile(q)
        )
        e["cnt"] += (e[f"centrality_{st}"] > cut).astype(int)

    e = e[e["cnt"] > 0].copy()
    e["target"] = (e["cnt"] >= 3).astype(int)

    d = e[["gene_name", "target"]].merge(hist_s, on="gene_name", how="inner")
    Xq, yq = d.drop(columns=["gene_name", "target"]), d["target"]

    pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", clone(best_models["RANDOM FOREST"]))
    ])

    s = cross_validate(pipe, Xq, yq, cv=cv, scoring=["roc_auc", "balanced_accuracy"])

    sens_rows.append({
        "cutoff": q,
        "n_genes": len(d),
        "pct_stable": round(100 * yq.mean(), 1),
        "cv_roc_auc": s["test_roc_auc"].mean(),
        "cv_roc_auc_sd": s["test_roc_auc"].std(),
        "cv_bal_acc": s["test_balanced_accuracy"].mean()
    })

sens = pd.DataFrame(sens_rows)
sens.to_csv(RESULTS / "hub_cutoff_sensitivity.csv", index=False)
print(sens.to_string(index=False, float_format="%.4f"))

 cutoff  n_genes  pct_stable  cv_roc_auc  cv_roc_auc_sd  cv_bal_acc
 0.8000     4509     63.5000      0.7022         0.0179      0.5971
 0.8500     3592     57.8000      0.6618         0.0269      0.5986
 0.9000     2545     51.0000      0.6711         0.0182      0.6179
 0.9500     1385     44.6000      0.7120         0.0454      0.6345


## SHAP

In [22]:
def positive_class_shap(shap_values):
    if isinstance(shap_values, list):
        return np.asarray(shap_values[1])
    sv = np.asarray(shap_values)
    return sv[:, :, 1] if sv.ndim == 3 else sv

def run_shap(model, X_te, name, top_n=20):
    tag = name.lower().replace(" ", "_")
    explainer = shap.TreeExplainer(model)
    shap_values = positive_class_shap(explainer.shap_values(X_te))

    importance = (pd.DataFrame({"feature": X_te.columns, "mean_abs_shap": np.abs(shap_values).mean(axis=0)})
                  .sort_values("mean_abs_shap", ascending=False).reset_index(drop=True))
    importance.to_csv(SHAP_DIR / f"{tag}_shap_importance.csv", index=False)
    print(f"\ntop {top_n} features ({name}):")
    print(importance.head(top_n).to_string(index=False))

    plt.figure()
    shap.summary_plot(shap_values, X_te, plot_type="bar", max_display=top_n, show=False)
    plt.title(f"SHAP importance - {name}")
    plt.tight_layout()
    plt.savefig(SHAP_DIR / f"{tag}_shap_bar.png", dpi=200, bbox_inches="tight")
    plt.close()
    return importance

rf_imp = run_shap(best_models["RANDOM FOREST"], X_test_s, "RANDOM FOREST")
xgb_imp = run_shap(best_models["XGBOOST"], X_test_s, "XGBOOST")


top 20 features (RANDOM FOREST):
      feature  mean_abs_shap
  H4K8ac_C14E       0.019494
 H3K4me3_C14E       0.012109
  H3K9ac_C14L       0.011345
 H3K18ac_C14E       0.010762
 H3K4me3_C14L       0.010704
 H3K27ac_C14E       0.010215
 H3K4me1_C14L       0.007986
 H3K4me1_C14E       0.007652
 H3K27ac_C14L       0.007530
  H4K5ac_C14L       0.007409
 H4K5ac_C9-13       0.007088
 H3K9ac_C9-13       0.007030
 H3K27ac_C1-8       0.006104
 H3K18ac_C1-8       0.005874
      H3_C14E       0.005411
     H3_C9-13       0.005119
H3K4me3_C9-13       0.005116
 H4K8ac_C9-13       0.005058
 H3K4me1_C1-8       0.005040
  H4K5ac_C14E       0.005008

top 20 features (XGBOOST):
      feature  mean_abs_shap
  H4K8ac_C14E       0.379884
  H3K9ac_C14L       0.285048
 H3K4me3_C14E       0.281987
 H3K4me3_C14L       0.266963
 H4K5ac_C9-13       0.259082
  H4K5ac_C14L       0.250546
 H3K18ac_C14E       0.246412
 H4K8ac_C9-13       0.237504
 H3K27ac_C14E       0.233335
     H3_C9-13       0.231946
 H3K9ac_C9

## Save metrics summary

In [23]:
summary = pd.DataFrame(all_metrics)
summary.to_csv(RESULTS / "results.csv", index=False)
print(summary[["model", "roc_auc", "balanced_accuracy", "f1_macro"]].to_string(index=False))

        model  roc_auc  balanced_accuracy  f1_macro
RANDOM FOREST 0.708122           0.604797  0.602622
      XGBOOST 0.680013           0.602288  0.603327
